In [16]:
import numpy as np
import scipy.io as sio
import scipy.stats as stats
import scipy.signal as signal
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

FS = 12_000
WINDOW_SIZE = FS
STEP_SIZE = FS // 2

In [17]:
def get_de_signal(path: str) -> np.ndarray:
    """Load .mat file and automatically extract the Drive End signal."""
    mat = sio.loadmat(Path(path), squeeze_me=True)
    de_keys = [k for k in mat.keys() if "DE" in k and "time" in k]
    if not de_keys:
        raise KeyError(f"No DE key found in {path}. Available: {list(mat.keys())}")
    return mat[de_keys[0]].astype(np.float64)

signals = {
    "Normal": get_de_signal("../data/normal/97.mat"),
    "Inner Race": get_de_signal("../data/12k drive end/105.mat"),
    "Ball": get_de_signal("../data/12k drive end/118.mat"),
    "Outer Race": get_de_signal("../data/12k drive end/130.mat"),
}

for name, sig in signals.items():
    print(f"{name:12s}: {len(sig):>7} samples = {len(sig)/FS:.1f}s")

Normal      :  243938 samples = 20.3s
Inner Race  :  121265 samples = 10.1s
Ball        :  122571 samples = 10.2s
Outer Race  :  121991 samples = 10.2s


In [18]:
def extract_features(sig: np.ndarray) -> dict:
    """Extract time-domain features from a single signal window"""
    rms = np.sqrt(np.mean(sig**2))
    kurtosis = stats.kurtosis(sig)
    crest_factor = np.max(np.abs(sig)) / rms
    peak_to_peak = np.max(sig) - np.min(sig)
    skewness = stats.skew(sig)
    return {
        "rms": rms,
        "kurtosis": kurtosis,
        "crest_factor": crest_factor,
        "peak_to_peak": peak_to_peak,
        "skewness": skewness,
    }

def build_feature_matrix(signals: dict, window_size: int, step_size: int) -> pd.DataFrame:
    """Slide a window over each Signal and extract features per window."""
    rows = []
    for label, sig in signals.items():
        starts = range(0, len(sig) - window_size, step_size)
        for start in starts:
            window = sig[start: start + window_size]
            features = extract_features(window)
            features["label"] = label
            rows.append(features)
    return pd.DataFrame(rows)

df_features = build_feature_matrix(signals, WINDOW_SIZE, STEP_SIZE)

print(f"Total windows: {len(df_features)}")
print(f"Per class:\n{df_features['label'].value_counts()}")

Total windows: 96
Per class:
label
Normal        39
Inner Race    19
Ball          19
Outer Race    19
Name: count, dtype: int64


In [27]:
def compute_spectrogram_patch(sig: np.ndarray, fs:int):
    """
    Compute a single STFT spectogram patch from a signal window.
    Returns a 2D array (frequency bins x time frames) in dB scale.
    """

    freqs, times, Zxx = signal.stft(
    sig, fs=fs,
    window = "hann",
    nperseg = 256,
    noverlap =192,
    boundary = None
    )

    freq_mask = freqs <= 3000
    power_db = 20 * np.log10(np.abs(Zxx[freq_mask]) + 1e-10)
    return power_db

def build_spectrogram_dataset(signals: dict, window_size: int, step_size:int, fs: int):
    """
    Build dataset of spectogram patches and corresponding labels.
    Returns X (patches) and y (integer labels).
    """
    label_map = {"Normal": 0, "Inner Race": 1, "Ball": 2, "Outer Race": 3}

    X, y = [], []

    for label, sig in signals.items():
        starts = range(0, len(sig) - window_size, step_size)
        for start in starts:
            window = sig[start : start + window_size]
            patch = compute_spectrogram_patch(window, fs)
            X.append(patch)
            y.append(label_map[label])

    X = np.array(X) # shape: (n_windows, freq_bins, time_frames)
    y = np.array(y)
    return X, y, label_map

X, y, label_map = build_spectrogram_dataset(signals, WINDOW_SIZE, STEP_SIZE, FS)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Label map: {label_map}")

X shape: (96, 65, 185)
y shape: (96,)
Label map: {'Normal': 0, 'Inner Race': 1, 'Ball': 2, 'Outer Race': 3}


In [28]:
# Saving feature matrix and spectrogram patches
np.save("../models/X_patches.npy", X)
np.save("../models/y_labels.npy", y)
df_features.to_csv("../models/features.csv", index=False)

print("Saved:")
print(f"X_patches.npy -> {X.shape}")
print(f"y_labels.npy -> {y.shape}")
print(f"features.csv -> {df_features.shape}")

Saved:
X_patches.npy -> (96, 65, 185)
y_labels.npy -> (96,)
features.csv -> (96, 6)
